# End-to-End Pipeline Demo

Runs the full agent-based model sequence on ~10 agents:
synthesize → persona → work/school zone → activities → destinations →
schedule → tours → mode choice.

**Requires** a `GEMINI_API_KEY` environment variable.

## 1. Imports + client setup

In [ ]:
from collections import Counter

from google import genai

from aibm import ZoneSpec, synthesize_population
from aibm.agent import ModeOption
from aibm.zone import Zone

client = genai.Client()
print("Client ready.")

## 2. Define zones

Three zones for our toy model:
- **north** — residential suburb
- **centre** — commercial downtown
- **east** — mixed: residential, commercial, and school

In [ ]:
zones = [
    Zone(
        id="north",
        name="North Suburb",
        x=0.0,
        y=1.0,
        land_use={"residential": True, "commercial": False, "school": False},
    ),
    Zone(
        id="centre",
        name="City Centre",
        x=0.0,
        y=0.0,
        land_use={"residential": False, "commercial": True, "school": False},
    ),
    Zone(
        id="east",
        name="East Side",
        x=1.0,
        y=0.0,
        land_use={"residential": True, "commercial": True, "school": True},
    ),
]

zone_lookup = {z.id: z for z in zones}
print(f"{len(zones)} zones defined.")

## 3. Define travel times

Dict mapping `(origin, destination)` to `{mode: minutes}`.

In [ ]:
travel_times = {
    ("north", "north"): {"car": 5, "transit": 10, "bike": 8, "walk": 15},
    ("north", "centre"): {"car": 15, "transit": 25, "bike": 30, "walk": 60},
    ("north", "east"): {"car": 20, "transit": 35, "bike": 40, "walk": 80},
    ("centre", "north"): {"car": 15, "transit": 25, "bike": 30, "walk": 60},
    ("centre", "centre"): {"car": 5, "transit": 8, "bike": 7, "walk": 12},
    ("centre", "east"): {"car": 10, "transit": 20, "bike": 25, "walk": 50},
    ("east", "north"): {"car": 20, "transit": 35, "bike": 40, "walk": 80},
    ("east", "centre"): {"car": 10, "transit": 20, "bike": 25, "walk": 50},
    ("east", "east"): {"car": 5, "transit": 10, "bike": 8, "walk": 15},
}


def get_travel_times_from(home: str) -> dict[str, dict[str, float]]:
    """Get travel times from a home zone to all other zones."""
    return {dest: modes for (orig, dest), modes in travel_times.items() if orig == home}


print("Travel times defined for all zone pairs.")

## 4. Synthesize ~10 agents

Two small zones, a few households each.

In [ ]:
zone_specs = [
    ZoneSpec(
        zone_id="north",
        n_households=3,
        household_size_dist={1: 0.2, 2: 0.5, 3: 0.3},
        age_dist={"0-17": 0.15, "18-64": 0.70, "65+": 0.15},
        employment_rate=0.70,
        student_rate=0.10,
        vehicle_dist={0: 0.1, 1: 0.6, 2: 0.3},
        license_rate=0.85,
    ),
    ZoneSpec(
        zone_id="east",
        n_households=3,
        household_size_dist={1: 0.3, 2: 0.4, 3: 0.3},
        age_dist={"0-17": 0.10, "18-64": 0.75, "65+": 0.15},
        employment_rate=0.50,
        student_rate=0.25,
        vehicle_dist={0: 0.4, 1: 0.45, 2: 0.15},
        license_rate=0.65,
    ),
]

households = synthesize_population(zone_specs, seed=42)
all_agents = [agent for hh in households for agent in hh.members]

print(f"Households: {len(households)}, Agents: {len(all_agents)}")
for hh in households:
    print(f"  {hh}  zone={hh.home_zone}  vehicles={hh.num_vehicles}")
    for a in hh.members:
        lic = "licence" if a.has_license else "no licence"
        print(f"    {a.name}  age={a.age}  {a.employment}  {lic}")

## 5. Generate personas

In [ ]:
hh_lookup = {a.id: hh for hh in households for a in hh.members}

for agent in all_agents:
    hh = hh_lookup[agent.id]
    agent.generate_persona(client=client, household=hh)
    print(f"{agent.name}: {agent.persona}")

## 6. Choose work/school zones

In [ ]:
for agent in all_agents:
    home = agent.home_zone
    assert home is not None
    tt = get_travel_times_from(home)

    if agent.employment == "employed":
        zone_id = agent.choose_work_zone(zones, tt, client=client)
        print(f"{agent.name} works in: {zone_id}")
    elif agent.employment == "student":
        zone_id = agent.choose_school_zone(zones, tt, client=client)
        print(f"{agent.name} studies in: {zone_id}")
    else:
        print(f"{agent.name}: no work/school zone needed ({agent.employment})")

## 7. Generate activities

In [ ]:
agent_activities: dict[str, list] = {}

for agent in all_agents:
    activities = agent.generate_activities(client=client)
    agent_activities[agent.id] = activities
    types = [a.type for a in activities]
    print(f"{agent.name}: {types}")

## 8. Choose destinations (flexible activities only)

In [ ]:
for agent in all_agents:
    for activity in agent_activities[agent.id]:
        if activity.is_flexible and activity.location is None:
            agent.choose_destination(activity, candidates=zones, client=client)
            print(f"{agent.name} → {activity.type} at {activity.location}")

## 9. Schedule activities

In [ ]:
agent_plans: dict[str, object] = {}

for agent in all_agents:
    activities = agent_activities[agent.id]
    day_plan = agent.schedule_activities(activities, client=client)
    agent_plans[agent.id] = day_plan
    print(f"\n{agent.name}'s schedule:")
    for act in day_plan.activities:
        print(
            f"  {act.type:12s}  "
            f"{act.start_time:6.0f}-{act.end_time:6.0f}  "
            f"@ {act.location}"
        )

## 10. Build tours

In [ ]:
for agent in all_agents:
    day_plan = agent_plans[agent.id]
    agent.build_tours(day_plan)
    n_tours = len(day_plan.tours)
    n_trips = len(day_plan.trips)
    print(f"\n{agent.name}: {n_tours} tour(s), {n_trips} trip(s)")
    for i, tour in enumerate(day_plan.tours):
        print(f"  Tour {i + 1} (closed={tour.is_closed}):")
        for trip in tour.trips:
            print(f"    {trip.origin} → {trip.destination}")

## 11. Choose mode per trip

In [ ]:
all_mode_choices = []

for agent in all_agents:
    hh = hh_lookup[agent.id]
    day_plan = agent_plans[agent.id]

    for trip in day_plan.trips:
        pair = (trip.origin, trip.destination)
        tt = travel_times.get(pair, {"walk": 30})

        # Build mode options, filtering car if no licence or 0 vehicles
        options = []
        for mode, minutes in tt.items():
            if mode == "car" and (not agent.has_license or hh.num_vehicles == 0):
                continue
            options.append(ModeOption(mode=mode, travel_time=minutes))

        if not options:
            options = [ModeOption(mode="walk", travel_time=30)]

        choice = agent.choose_mode(options, client=client, household=hh)
        trip.mode = choice.option.mode
        all_mode_choices.append(choice)
        print(
            f"{agent.name}: {trip.origin}→{trip.destination} "
            f"by {trip.mode} ({choice.option.travel_time:.0f} min)"
        )

## 12. Summary — mode share

In [ ]:
all_trips = [trip for agent in all_agents for trip in agent_plans[agent.id].trips]

mode_counts = Counter(t.mode for t in all_trips)
total = len(all_trips)

print(f"Total trips: {total}\n")
print("Mode share:")
for mode, count in mode_counts.most_common():
    print(f"  {mode:8s}: {count:3d}  ({count / total:.0%})")